**Подготовка файла 1100.csv на основе данных сводных файлов 2016-2018 годов**

- Автор: Клинкова Екатерина
- Telegram: @ekaklinkova

- `..\Data\{год}\` - папка для хранения созданнного 2100.csv для определенного года
- `..\Data\{год}\Original` - папка для хранения оригинального файла xlsx
- `..\Data\Common\` - папка для хранения итогового результата 2100.csv сводный

Пример: 
- *E:/IrkutskProject/Data/2016/Original/ф33  правл за 2016г Иркутская обл.xlsx*
- *E:/IrkutskProject/Data/2016/2100.csv*

In [4]:
#!pip install python-docx
#!pip install Document

In [5]:
# Загрузка библиотек
#from docx import Document
import pandas as pd
import numpy as np
import os
import docx
import re

In [6]:
path = 'E:/IrkutskProject/Data/'
# Список годов для обработки
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Название оригинального файла
original_names = {
    2016: 'ИО 30 2016.docx',
    2017: 'ИО 30 2017.docx',
    2018: 'ИО 30 2018.docx',
    2019: 'ИО 30 2019.docx',
    2020: 'ИО 30 2020.docx',
    2021: '30 ИО 2021.docx',
    2022: '30 ИО 2022.docx',
    2023: '30 ИО 2023.docx',
    2024: '30 ИО 2024.docx'
}

original_url = {}
for year in years:
    original_path = f'{path}{year}/Original/'
    original_url[year] = original_path + original_names[year]
    
    if not os.path.exists(original_url[year]):
        print(f"Файл {original_url[year]} не найден по указанному пути")
    else:
        print(f"Файл {original_url[year]} найден по указанному пути")

Файл E:/IrkutskProject/Data/2016/Original/ИО 30 2016.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2017/Original/ИО 30 2017.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2018/Original/ИО 30 2018.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2019/Original/ИО 30 2019.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2020/Original/ИО 30 2020.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2021/Original/30 ИО 2021.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2022/Original/30 ИО 2022.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2023/Original/30 ИО 2023.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2024/Original/30 ИО 2024.docx найден по указанному пути


In [7]:
def extract_tuberculosis_table(file_path, target_block):
    doc = docx.Document(file_path)
    target_found = False
    table_index = -1
    selected_tables = []

    # Перебираем все элементы в теле документа
    for i, element in enumerate(doc.element.body):
        # Шаг 1: Ищем текст заголовка
        #if not target_found and element.tag.endswith('p'):
        if element.tag.endswith('p'):
            text = "".join(element.itertext()).strip()
            # display(text)
            if target_block.lower() in text.lower():
                target_found = True
                print(f"Текст '{target_block}' найден. Ищем следующую таблицу...")
            if '1101' in text.lower():
                print(f"Текст 1101 найден. Таблица закончилась")
                break

        # Шаг 2: После нахождения текста ищем ПЕРВУЮ встречную таблицу
        if target_found and element.tag.endswith('tbl'):
            # Считаем количество таблиц до текущего элемента
            table_count = len([e for e in doc.element.body[:i] if e.tag.endswith('tbl')])
            table_index = table_count
            selected_tables.append(doc.tables[table_index])
            print(f"Таблица найдена (индекс в документе: {table_index})")
            #break  # Выходим после нахождения первой таблицы

    if not target_found:
        raise ValueError(f"Текст '{target_block}' не найден в документе")
    elif table_index == -1:
        raise ValueError(f"Текст найден, но после него нет ни одной таблицы")

    # Возвращаем всю таблицу (включая все страницы)
    # return doc.tables[table_index]
    return selected_tables

def table_to_dataframe(table):
    data = []
    for row in table.rows:
        current_row = []
        
        for cell in row.cells:
            text = cell.text.strip()
            current_row.append(text)
        data.append(current_row)

    # Создаём DataFrame
    if data:  # Проверяем, что таблица не пустая
        df = pd.DataFrame(data[1:], columns=data[0])
    else:
        df = pd.DataFrame()  # Пустой DataFrame, если таблица пустая
    return df

In [8]:
try:
    df_1100 = {}
    target_block = "1100"
    selected_tables = []
    for year in years:
        display(year)
        #table = extract_tuberculosis_table(original_url[year], target_block)
        selected_tables = extract_tuberculosis_table(original_url[year], target_block)
        # df_1100[year] = table_to_dataframe(table)
        for table in selected_tables:
            df = table_to_dataframe(table)
            
            df_1100[year] = df[
                (df['№ стр.'] == '2') |
                (df['№ стр.'] == '110') |
                (df['№ стр.'] == '111') | 
                (df['№ стр.'] == '112')
            ]
            #df_1100[year] = df[
            #    (df['Наименование должности\n(специальности)'].str.contains('фтизиатры'))
            #]

            if df_1100[year].shape[0] > 1:
                break
        df_1100[year].to_csv(f'E:/IrkutskProject/Data/{year}/1100.csv', index=False, encoding='utf-8-sig')
except Exception as e:
    print(f"Произошла ошибка: {str(e)}")

2016

Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 26)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 27)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 28)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 29)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 30)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 31)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 32)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 33)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 34)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 35)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 36)
Текст '1100' найден. Ищем следую

2017

Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 26)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 27)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 28)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 29)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 30)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 31)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 32)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 33)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 34)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 35)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 36)
Текст '1100' найден. Ищем следую

2018

...

Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 27)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 28)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 29)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 30)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 31)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 32)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 33)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 34)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 35)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 36)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 37)
Текст '1100' найден. Ищем следую

2023

Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 27)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 28)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 29)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 30)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 31)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 32)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 33)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 34)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 35)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 36)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 37)
Текст '1100' найден. Ищем следую

2024

Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 29)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 30)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 31)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 32)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 33)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 34)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 35)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 36)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 37)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 38)
Текст '1100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 39)
Текст '1100' найден. Ищем следую

In [9]:
for year in years:
    if ((year == 2016) or (year == 2017) or (year == 2019) or (year == 2021)):
        df_1100[year].drop(df_1100[year].index[3:], inplace=True)
    else:
        df_1100[year].drop(df_1100[year].index[1:2], inplace=True)
    df_1100[year].iloc[1,0] = 'фтизиатры'
    df_1100[year].iloc[2,0] = 'из них фтизиатры участковые'
    df_1100[year] = df_1100[year].drop(df_1100[year].columns[9:], axis=1)
    df_1100[year].columns = [
        'Наименование должности',
        'Номер строки',
        'Число должностей в целом по организации, штатных',
        'Число должностей в целом по организации, занятых',
        'Число физических лиц основных работников на занятых должностях'
    ]
    display(year)
    display(df_1100[year])

2016

,Наименование должности,Номер строки,"Число должностей в целом по организации, штатных","Число должностей в целом по организации, занятых",Число физических лиц основных работников на занятых должностях
2,1,2,3,4,9
12,фтизиатры,110,"209,00","197,75",119
13,из них фтизиатры участковые,111,"75,00","73,25",36


2017

,Наименование должности,Номер строки,"Число должностей в целом по организации, штатных","Число должностей в целом по организации, занятых",Число физических лиц основных работников на занятых должностях
2,1,2,3,4,9
12,фтизиатры,110,"246,25","229,25",132
13,из них фтизиатры участковые,111,"78,25","75,50",36


2018

...

2023

,Наименование должности,Номер строки,"Число должностей в целом по организации, штатных","Число должностей в целом по организации, занятых",Число физических лиц основных работников на занятых должностях
2,1,2,3,4,9
6,фтизиатры,111,"229,50","193,25",136
7,из них фтизиатры участковые,112,"70,75","64,25",44


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18876\4103348389.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1100[year].drop(df_1100[year].index[1:2], inplace=True)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18876\4103348389.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1100[year].iloc[1,0] = 'фтизиатры'
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18876\4103348389.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  

2024

,Наименование должности,Номер строки,"Число должностей в целом по организации, штатных","Число должностей в целом по организации, занятых",Число физических лиц основных работников на занятых должностях
2,1,2,3,4,9
6,фтизиатры,111,"201,50","173,75",121
7,из них фтизиатры участковые,112,"71,25","66,50",46


In [10]:
columns = [
    'Показатель',
    'Уточнение',
    'Год',
    'Значение',
    'Строка',
    'Графа',
    'Таблица'
]

data = []
data_2016_2018 = []

df_1100_long = pd.DataFrame(columns=columns)
df_1100_long_2016_2018 = pd.DataFrame(columns=columns)

for year in years:
    for row_number in range(1, df_1100[year].shape[0]):
        for col_number in range(2, df_1100[year].shape[1]):
                new_row = []
                new_row.append(df_1100[year].iloc[row_number, 0]) # Показатель
                new_row.append(df_1100[year].columns[col_number]) # Уточнение
                new_row.append(year)
                new_row.append((df_1100[year].iloc[row_number, col_number]).replace(',', '.')) # Значение
                new_row.append(df_1100[year].iloc[row_number,1]) # Строка
                new_row.append(df_1100[year].iloc[0, col_number]) # Графа
                new_row.append(1100)
                data.append(new_row)
                if year <= 2018:
                   data_2016_2018.append(new_row)

df_1100_long = pd.DataFrame(data, columns=columns)
df_1100_long_2016_2018 = pd.DataFrame(data_2016_2018, columns=columns)

In [11]:
df_1100_long_2016_2018.to_csv(f'{path}Common_2016-2018/1100_2016-2018.csv', index=False)
df_1100_long.to_csv(f'{path}Common/1100_2016-2024.csv', index=False)